<a href="https://colab.research.google.com/github/ConstantinaGio/aip-email-marketing_Group_32/blob/main/IB9BW0_Analytics_in_Practice_Group_32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IB9BW0 Analytics in Practice
## Group Assignment: E-Mail Marketing for Universal Plus
**Group:** 32

---

### Contents
1. Business Understanding
2. Data Understanding
3. Data Preparation

**Note:** This notebook runs from start to finish using only `campaign_data.csv` and `next_campaign.csv`, placed in the same folder as the notebook.

In [1]:
# upload campaign_data.csv and next_campaign.csv
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

TypeError: 'NoneType' object is not subscriptable

-----
## 1. Business Understanding

***Business problem:*** Universal Plus sends campaign e-mails mostly at random or by rules of thumb.Sending e-mails to uninterested customers costs money and several past campaigns were not successful.

***Analytics problem:*** predict whether each customer will visit the shop in the two weeks after a campaign, using only information available before the campaign. Because the answer is a category (yes or no), this is a *classification* problem.

**Key figures from the board:**
- Each e-mail costs **£3.50**.
- A visiting customer spends £93 on average, with a 30% margin, so each visit is worth about **£28**.



In [ ]:
# Libraries we use in this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Figures given by the board
email_cost = 3.50
visit_value = 93 * 0.30          # average spend x margin
break_even = email_cost / visit_value

print(f"Value of a visit: £{visit_value:.2f}")
print(f"Break-even: {break_even:.1%}")

-----
## 2. Data Understanding

### 2.1 Loading the data

In [ ]:
# Read both files (only empty cells count as missing)
campaign = pd.read_csv("campaign_data.csv", keep_default_na=False, na_values=[""])
next_campaign = pd.read_csv("next_campaign.csv", keep_default_na=False, na_values=[""])

# Number of rows and columns
print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)

In [ ]:
# The first five customers
campaign.head()

In [ ]:
# The first five customers in the next campaign file
next_campaign.head()

In [ ]:
# The structure: column names, data types and the number of non-missing values
campaign.info()

**Findings:**
- The campaign data contains 52,000 customers from the first campaign and 21 columns. The next campaign file contains 12,000 customers and 20 columns, as the `visit` column has been excluded.
- `Customer_ID` is a unique identifier and will therefore be removed before modelling.
- Four columns are stored as text (`purchase_segment`, `zip_area`, `channel`, `email_segment`), so they will need encoding.
- Some columns have fewer than 52,000 non-null values, so there are missing values to check and handle.

### 2.2 Missing values


In [ ]:
# Count the missing values in each column and show them as a percentage,
# sorted so the column with the most missing values appears first.

# Number of missing values per column (isnull() and sum())
missing_count = campaign.isnull().sum()

# Share of missing values per column (%)
missing_percent = (campaign.isnull().sum() / len(campaign) * 100).round(1)

# Create a summary table with both number and percentage of missing values
missing_table = pd.DataFrame({"missing": missing_count, "percent": missing_percent})
missing_table[missing_table["missing"] > 0].sort_values("percent", ascending=False)

In [ ]:
# The same missing value check applied to the next campaign file (%)
(next_campaign.isnull().sum() / len(next_campaign) * 100).round(1)

**Findings:**
- `satisfaction_score` has the most missing values: about 60% in both files. This is because only customers who answered the survey have a score. *(readme.txt)*
- The other columns have a percentage of small missing values: `recency` (7.2%), `age` (3.9%), and less than 2% for `channel`, `zip_area`, `spend` and `purchase_segment`.
- If we drop every missing value, we would lose about 60% of the customers and introduce heavy bias into our model. So instead, we will fill the gaps.

Since both files follow the same pattern, they can be cleaned the same way


### 2.3 Data quality checks


In [ ]:
# account: do all customers have an account? (1 = yes, 0 = no)
print(campaign["account"].value_counts(), "\n")

# employed: are almost all customers employed? (1 = yes, 0 = no)
print(campaign["employed"].value_counts(), "\n")

# spend: do only customers who visited spend money? (True = spent money)
print(pd.crosstab(campaign["spend"] > 0, campaign["visit"]), "\n")

# purchase_segment: do the price groups match the purchase amounts?
print(campaign.groupby("purchase_segment")["purchase"].agg(["min", "max"]), "\n")

# Are any values impossible? (min and max of the number columns)
print(campaign[["recency", "purchase", "age", "satisfaction_score"]].describe().round(1), "\n")

# Are there any duplicated customers?
print("Duplicate customers:", campaign["Customer_ID"].duplicated().sum())

In [ ]:
# employed: how many customers are not employed, and do they visit differently?
# employed: 1 = has a permanent job, 0 = does not

print(campaign.groupby("employed")["visit"].agg(["count", "mean"]).round(3), "\n")

**Findings:**
- `Customer_ID`: is just a label/identifier. Columns that only identify customers are not real information, so we remove it.
- `spend`: this is the total amount a customer spent in following two weeks after the campaign. Only customers who visited the shop spent money and we will know the amount spend after the campaign --> we cannot use it to decide who to e-mail. This is data leakage (Kapoor and Narayanan, 2023), so we remove it.
- `account`: every customer has an account,so this column gives the model no information, so we remove it.
- `employed`: Almost all customers are employed, with only 86 not employed. Non-employed customers visit the shop less often (9.3% compared with 16.4%). However, there are too few non-employed customers for the model to learn from, so it will be dropped.
- `purchase_segment`: it's the same as `purchase`, just in price groups, so it adds nothing new. We remove it.
- All values are realistic (e.g. age 19–62, recency 1–12 months) and there are no duplicate customers. Duplicates could put the same customer in both the training and test set, which is another type of leakage (Kapoor and Narayanan, 2023).


-----
## 3. Data Preparation

### 3.1 Removing columns
Based on the checks in 2.3, we remove the 5 columns that do not help the model or give away the answer.

In [ ]:
# Columns to remove
drop_columns = ["Customer_ID", "spend", "account", "employed", "purchase_segment"]

# Remove them from both files, so they are the same
campaign = campaign.drop(columns=drop_columns)
next_campaign = next_campaign.drop(columns=drop_columns)

# Check the new size
print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)


### 3.2 Handling missing values

We do not delete rows, because we would lose about 60% of customers (see 2.2). Instead, we will fill those missing values:

| Column | Missing (%) | What we do | Why |
|---|---|---|---|
| `satisfaction_score` | 59.8 | Add a missing flag, then fill with the median | Only customers who answered the survey have a score, so a missing score tells us something |
| `recency` | 7.2 | Add a missing flag, then fill with the median | The median is the middle value, so a few very high or low numbers don't change it |
| `age` | 3.9 | Add a missing flag, then fill with the median | The median is the middle value, so a few very high or low numbers don't change it |
| `zip_area` | 1.5 | Fill with a new category "Missing"|Mark it as missing instead of guessing
| `channel` | 1.5 | Fill with a new category "Missing" | Mark it as missing instead of guessing |


The median is filled in after the train/test split, so it only comes from the training data.

In [ ]:
# Add a missing flag for each number column (1 = was missing, 0 = not missing)
# As in Demo 1, we add the flags before filling, so we keep a record of the gaps
for column in ["recency", "age", "satisfaction_score"]:
    campaign[column + "_missing"] = campaign[column].isnull().astype(int)
    next_campaign[column + "_missing"] = next_campaign[column].isnull().astype(int)

In [ ]:
# Do customers without a satisfaction score visit less often? (visit rate %)
# satisfaction_score_missing: 1 = no score, 0 = has a score
(campaign.groupby("satisfaction_score_missing")["visit"].mean() * 100).round(1)

In [ ]:
# Fill the gaps in the text columns with a new category "Missing"
for column in ["zip_area", "channel"]:
    campaign[column] = campaign[column].fillna("Missing")
    next_campaign[column] = next_campaign[column].fillna("Missing")

# Check: which columns still have gaps?
print(campaign.isnull().sum()[campaign.isnull().sum() > 0])

**Findings:**
- We added three missing flags: `recency_missing`, `age_missing` and `satisfaction_score_missing`.
- Customers without a satisfaction score visit less often (13.7%) than customers with a score (20.4%). So whether a customer answered the survey is useful information.These will be filled with the median. The flag (1 = did not answer) helps the model understand which customers did not answer the survey.
- The gaps in `zip_area` and `channel` are now a "Missing" category.
- Only `recency`, `age` and `satisfaction_score` still have gaps. We fill these with the median after the train/test split.

### 3.3 Encoding
We need to turn all text types into binary (0/1) columns as models only work with numbers:
- `email_segment`: two new columns, `mens_email` and `womens_email` (1 = received it, 0 = did not). If both are 0, the customer got no e-mail.
- `zip_area`, `channel`, `marriage` and `delivery` have no natural order, so we use one-hot encoding.
-  `marriage` and `delivery` are stored as numbers, but they are groups (e.g. 1 = married, 2 = single), so they need to be encoded.

In [ ]:
# E-mail received: two 0/1 columns, using np.where() as in Demo 2
# No e-mail = 0 in both columns
for data in [campaign, next_campaign]:
    data["mens_email"] = np.where(data["email_segment"] == "Mens E-Mail", 1, 0)
    data["womens_email"] = np.where(data["email_segment"] == "Womens E-Mail", 1, 0)

# Remove the original text column
campaign = campaign.drop(columns=["email_segment"])
next_campaign = next_campaign.drop(columns=["email_segment"])

In [ ]:
# One 0/1 column per group (one-hot encoding, as in Demo 2)
encode_columns = ["zip_area", "channel", "marriage", "delivery"]

campaign = pd.get_dummies(campaign, columns=encode_columns, drop_first=True, dtype=int)
next_campaign = pd.get_dummies(next_campaign, columns=encode_columns, drop_first=True, dtype=int)

print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)

In [ ]:
# Every column should now be a number
campaign.info()

In [ ]:
# Check 1: the new e-mail columns (1 = got this e-mail, both 0 = no e-mail)
print(campaign[["mens_email", "womens_email"]].head(), "\n")

# Check 2: list the 0/1 columns created by one-hot encoding
new_columns = [c for c in campaign.columns if c.startswith(("zip_area_", "channel_", "marriage_", "delivery_"))]
print("New 0/1 columns:", new_columns, "\n")

# Check 3: make sure there are no text columns left (only numbers)
print(campaign.dtypes.value_counts(), "\n")

# Check 4: both files should have the same columns (apart from visit)
print("Same columns:", list(campaign.drop(columns=["visit"]).columns) == list(next_campaign.columns))

**Findings:**
- The e-mail groups are now two 0/1 columns: `mens_email` and `womens_email`.
- `zip_area`, `channel`, `marriage` and `delivery` are now 0/1 columns, one for each group.
- All columns are now numbers, and both files have exactly the same columns.

Therefore, the data is ready to be split into train/test sets.



### 3.4 Train/test split and filling the remaining gaps

The data will be split into:
-  a training set (70%), used to build the models
-  a test set (30%), to check how well models predict customers.

We use `stratify=y` so both sets have the same share of visitors.

After the split, we fill the gaps in `recency`, `age` and `satisfaction_score` with the median. We only use the training set to work out the median, then use the same numbers for the test set and the next campaign.

Filling  the gaps before splitting, the test customers help calculate the median, so they influence the training data. If we split first and use only the training data, the test stays untouched. Filling gaps using the whole dataset would indroduce data leakage (Kapoor and Narayanan, 2023). This keeps the test set separate, so the test results show how the models would work on new customers



In [ ]:
from sklearn.model_selection import train_test_split

# Features (everything except the target) and target (visit)
X = campaign.drop(columns=["visit"])
y = campaign["visit"]

# 70% training, 30% test (as in Demo 2)
# stratify=y keeps the share of visitors the same in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10, stratify=y)

print("Training set:", X_train.shape[0], "customers")
print("Test set:    ", X_test.shape[0], "customers")

# Share of visitors in each set
print("\nAll data:\n", y.value_counts(normalize=True).round(3))
print("\nTraining set:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest set:\n", y_test.value_counts(normalize=True).round(3))

In [ ]:
# Median of each column, worked out from the training set only
fill_columns = ["recency", "age", "satisfaction_score"]
medians = X_train[fill_columns].median()
print("Medians from the training set:\n", medians)

# Use the same medians for all three datasets
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)
X_next = next_campaign.fillna(medians)

# Check: no gaps left anywhere
print("\nMissing values left:")
print("Training set:", X_train.isnull().sum().sum())
print("Test set:", X_test.isnull().sum().sum())
print("Next campaign:", X_next.isnull().sum().sum())

In [ ]:
# Check: are the training medians the same as the medians of all the data?
median_check = pd.DataFrame({
    "Training set": medians,
    "All data": X[fill_columns].median(),
})
median_check

In [ ]:
# How many values were filled in each dataset, and with which median?
# The missing flags (1 = was missing) tell us how many gaps were filled
filled_table = pd.DataFrame({
    "Filled with (median)": medians,
    "Filled in training set": [X_train[c + "_missing"].sum() for c in fill_columns],
    "Filled in test set": [X_test[c + "_missing"].sum() for c in fill_columns],
    "Filled in next campaign": [X_next[c + "_missing"].sum() for c in fill_columns],
})
filled_table

**Findings:**
- Training set: 36,400 customers. Test set: 15,600 customers.
- Both sets have the same share of visitors (16.4%).
- Since the medians from the training set are the same as for all the data (recency 6, age 34, satisfaction score 7). So filling after the split does not change the results, but it keeps the test set separate.



-----
### References
1. Kapoor, S. and Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. Patterns, 4(9), pp.100804–100804. doi:10.1016/j.patter.2023.100804.